In [1]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

In [2]:
PROJECT_ROOT = Path(r"D:\Final_GRAG")
GRI_UNITS_CSV = PROJECT_ROOT / "metadata" / "gri_units" / "gri_units.csv"
GRI_UNITS_JSON = PROJECT_ROOT / "metadata" / "gri_units" / "gri_units.json"
OUTPUT_DIR = PROJECT_ROOT / "metadata" / "gri_units"

In [3]:
df = pd.read_csv(GRI_UNITS_CSV)

## Knowledge Generation

In [4]:
KNOWLEDGE_PROMPT = """You are a GRI Standards expert. For the following GRI requirement, generate structured knowledge extensions.

## GRI REQUIREMENT
- Standard: {standard_name} ({standard_id})
- Disclosure: {disclosure_name} ({disclosure_id})
- Requirement ID: {requirement_id}
- Requirement text: {requirement_text}
- Requirement type: {requirement_type}

## INSTRUCTIONS
Generate the following 5 sections. Use the EXACT section headers shown below — copy them letter-for-letter, including underscores (not spaces).

[EXPLANATION]
A clear, plain-language explanation of what this requirement asks organizations to disclose (2-3 sentences).

[COMPLIANCE_CRITERIA]
Specific criteria that determine whether a disclosure meets this requirement. List as bullet points starting with "- ". Provide at least 3 bullet points.

[METRICS]
If this involves quantitative data, list the typical metrics, units, and measurement methods as comma-separated items. If purely qualitative, write "N/A".

[SEARCH_TERMS]
List 8-10 keywords or phrases commonly found in sustainability reports that relate to this requirement. Separate with commas.

[BANKING_CONTEXT]
How this requirement specifically applies to the banking and financial services sector (2-3 sentences). If not particularly sector-specific, describe general applicability.

CRITICAL RULES:
1. You MUST include ALL five section headers exactly as shown: [EXPLANATION], [COMPLIANCE_CRITERIA], [METRICS], [SEARCH_TERMS], [BANKING_CONTEXT]
2. Use underscores, NOT spaces, in multi-word headers (COMPLIANCE_CRITERIA, SEARCH_TERMS, BANKING_CONTEXT)
3. Do NOT skip any section. Do NOT rename any section."""


def format_prompt(row):
    return KNOWLEDGE_PROMPT.format(
        standard_name=row["standard_name"],
        standard_id=row["standard_id"],
        disclosure_name=row["disclosure_name"],
        disclosure_id=row["disclosure_id"],
        requirement_id=row["requirement_id"],
        requirement_text=row["requirement_text"],
        requirement_type=row["requirement_type"],
    )

In [5]:
test = df.iloc[0]
print(format_prompt(test))

You are a GRI Standards expert. For the following GRI requirement, generate structured knowledge extensions.

## GRI REQUIREMENT
- Standard: Biodiversity (GRI 101)
- Disclosure: Policies to halt and reverse (101-1)
- Requirement ID: a
- Requirement text: describe its policies or commitments to halt and reverse biodiversity loss, and how these are informed by the 2050 Goals and 2030 Targets in the Kunming-Montreal Global Biodiversity Framework;
- Requirement type: shall

## INSTRUCTIONS
Generate the following 5 sections. Use the EXACT section headers shown below — copy them letter-for-letter, including underscores (not spaces).

[EXPLANATION]
A clear, plain-language explanation of what this requirement asks organizations to disclose (2-3 sentences).

[COMPLIANCE_CRITERIA]
Specific criteria that determine whether a disclosure meets this requirement. List as bullet points starting with "- ". Provide at least 3 bullet points.

[METRICS]
If this involves quantitative data, list the typical 

In [6]:
import ollama

def generate_knowledge(prompt):
    response = ollama.chat(
        model="llama3.1:8b",
        messages=[{"role": "user", "content": prompt}],
        options={
            "temperature": 0.2,
            "num_ctx": 4096,
            "top_p": 0.9,
        },
    )
    return response["message"]["content"]

test = generate_knowledge("Hello")
print(test[:100])

Hello! How are you today? Is there something I can help you with or would you like to chat?


In [7]:
test = df["standard_id"].str.contains("305", na=False)
test_row = df[test].iloc[0]
test_prompt = format_prompt(test_row)
test_response = generate_knowledge(test_prompt)

print(test_response)

Here are the generated structured knowledge extensions for GRI 305-1:

[EXPLANATION]
This requirement asks organizations to disclose their gross direct (Scope 1) greenhouse gas emissions in metric tons of CO2 equivalent, which includes emissions from sources owned or controlled by the organization.

[COMPLIANCE_CRITERIA]
- The disclosure must be based on a comprehensive and transparent calculation methodology.
- The calculation must include all direct GHG emissions from sources within the organization's control, such as fuel combustion, industrial processes, and fugitive emissions.
- The disclosure must be presented in metric tons of CO2 equivalent (MtCO2e) to facilitate comparison with other organizations.

[METRICS]
GHG emissions, metric tons of CO2 equivalent (MtCO2e), calculation methodology, ISO 14064-1:2006 or equivalent standard.

[SEARCH_TERMS]
greenhouse gas emissions, direct emissions, Scope 1 emissions, GHG footprint, carbon emissions, climate change, sustainability reportin

## 7. Parse LLM Response into Structured Fields

Regex-based parser that extracts each `[SECTION]` block into a dictionary.

In [8]:
SECTION_HEADERS = [
    "EXPLANATION",
    "COMPLIANCE_CRITERIA",
    "METRICS",
    "SEARCH_TERMS",
    "BANKING_CONTEXT",
]

SECTION_RE = re.compile(
    # Line start + optional whitespace
    r"^\s*"
    # Optional leading decorators: **, ***, ##, ### etc. in any order
    r"(?:\*{1,3})?"
    r"(?:#{1,3}\s*)?"
    r"(?:\*{1,3})?"
    # Optional opening bracket
    r"(?:\[)?"
    # Capture the section name (allow space or underscore)
    r"(" + "|".join(SECTION_HEADERS) + r")"
    # Optional closing bracket, trailing **, *, :
    r"(?:\])?"
    r"(?:\*{1,3})?"
    r":?\s*$",
    re.MULTILINE | re.IGNORECASE,
)

def to_list(text):
    if not text or text.strip().upper() == "N/A":
        return []
    items = re.split(r"[,\n]+", text)
    cleaned = []
    for item in items:
        item = re.sub(r"^[\s\-\*•\d.]+", "", item).strip().strip('"').strip("'")
        if item:
            cleaned.append(item)
    return cleaned

def parse_knowledge_response(text):
    sections = {}
    matches = list(SECTION_RE.finditer(text))

    for i, match in enumerate(matches):
        key = match.group(1).upper().replace(" ", "_")
        start = match.end()
        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(text)
        sections[key] = text[start:end].strip()

    return {
        "knowledge_text": sections.get("EXPLANATION", "").strip(),
        "compliance_criteria": sections.get("COMPLIANCE_CRITERIA", "").strip(),
        "metrics": to_list(sections.get("METRICS", "")),
        "search_terms": to_list(sections.get("SEARCH_TERMS", "")),
        "banking_context": sections.get("BANKING_CONTEXT", "").strip(),
        "raw_response": text,
    }

In [9]:
# Test
parsed = parse_knowledge_response(test_response)
for k, v in parsed.items():
    if k == "raw_response":
        continue
    print(f"{k} : {str(v)}")

knowledge_text : This requirement asks organizations to disclose their gross direct (Scope 1) greenhouse gas emissions in metric tons of CO2 equivalent, which includes emissions from sources owned or controlled by the organization.
compliance_criteria : - The disclosure must be based on a comprehensive and transparent calculation methodology.
- The calculation must include all direct GHG emissions from sources within the organization's control, such as fuel combustion, industrial processes, and fugitive emissions.
- The disclosure must be presented in metric tons of CO2 equivalent (MtCO2e) to facilitate comparison with other organizations.
metrics : ['GHG emissions', 'metric tons of CO2 equivalent (MtCO2e)', 'calculation methodology', 'ISO 14064-1:2006 or equivalent standard.']
search_terms : ['greenhouse gas emissions', 'direct emissions', 'Scope 1 emissions', 'GHG footprint', 'carbon emissions', 'climate change', 'sustainability reporting', 'environmental impact', 'emissions reductio

## Batch Generate Knowledge 

In [10]:
knowledge_results = {}

remain = [
    i for i in range(len(df)) if df.iloc[i]["unit_id"] not in knowledge_results
]

print(f"{len(remain)} / {len(df)}")

pbar = tqdm(remain, desc="Generating knowledge", unit="req")
for i in pbar:
    row = df.iloc[i]
    unit_id = row["unit_id"]

    prompt = format_prompt(row)
    raw_response = generate_knowledge(prompt)
    parsed = parse_knowledge_response(raw_response)
    parsed["unit_id"] = unit_id
    knowledge_results[unit_id] = parsed

918 / 918


Generating knowledge:   0%|          | 0/918 [00:00<?, ?req/s]

## Add knowledge to gri_units

In [11]:
# df cho knowledge_results 
knowledge_df = pd.DataFrame.from_dict(knowledge_results, orient="index")

# Đảm bảo unit_id không bị trùng
if "unit_id" in knowledge_df.columns:
    knowledge_df = knowledge_df.drop(columns=["unit_id"])
knowledge_df.index.name = "unit_id"
knowledge_df = knowledge_df.reset_index()

# list -> json string
knowledge_df["search_terms_clone"] = knowledge_df["search_terms"].apply(json.dumps)
knowledge_df["metrics_clone"] = knowledge_df["metrics"].apply(json.dumps)

# Merge
join_cols = [
    "unit_id",
    "knowledge_text",
    "compliance_criteria",
    "search_terms_clone",
    "metrics_clone",
    "banking_context",
]

df_joined = df.merge(
    knowledge_df[join_cols],
    on="unit_id",
    how="left",
)

# Rename cols
df_joined = df_joined.rename(columns={
    "search_terms_clone": "search_terms",
    "metrics_clone": "metrics",
})

In [12]:
df_joined[["unit_id", "knowledge_text", "search_terms", "banking_context"]].head()

,unit_id,knowledge_text,search_terms,banking_context
0,GRI101-2024-D101-1-Ra,This requirement asks organizations to disclos...,"[""biodiversity"", ""conservation"", ""sustainabili...","This requirement applies to all organizations,..."
1,GRI101-2024-D101-1-Rb,This requirement asks organizations to disclos...,"[""policies"", ""biodiversity loss"", ""halt and re...","In the banking and financial services sector, ..."
2,GRI101-2024-D101-1-Rc,This requirement asks organizations to disclos...,"[""biodiversity loss"", ""halt and reverse biodiv...",This requirement applies to the banking and fi...
3,GRI101-2024-D101-2-Ra,This requirement asks organizations to disclos...,"[""mitigation hierarchy"", ""biodiversity impacts...",This requirement applies to the banking and fi...
4,GRI101-2024-D101-2-Ra-i,This requirement asks organizations to disclos...,"[""biodiversity management"", ""habitat conservat...","In the banking and financial services sector, ..."


## Export extended gri_units

In [13]:
csv_out = OUTPUT_DIR / "gri_units_with_knowledge.csv"

df_joined.to_csv(csv_out, index=False, encoding="utf-8-sig")

In [14]:
json_out = OUTPUT_DIR / "gri_units_with_knowledge.json"

with open(GRI_UNITS_JSON, "r", encoding="utf-8") as f:
    original_units = json.load(f)

for unit in original_units:
    uid = unit["unit_id"]
    if uid in knowledge_results:
        k = knowledge_results[uid]
        unit["knowledge_text"] = k.get("knowledge_text", "")
        unit["compliance_criteria"] = k.get("compliance_criteria", "")
        unit["banking_context"] = k.get("banking_context", "")
        unit["search_terms"] = k.get("search_terms", [])
        unit["metrics"] = k.get("metrics", [])
    else:
        unit["knowledge_text"] = ""
        unit["compliance_criteria"] = ""
        unit["banking_context"] = ""
        unit["search_terms"] = []
        unit["metrics"] = []

with open(json_out, "w", encoding="utf-8") as f:
    json.dump(original_units, f, ensure_ascii=False, indent=2)

## Upload to Zilliz vector database

In [15]:
from dotenv import load_dotenv
from pymilvus import MilvusClient, DataType

load_dotenv(PROJECT_ROOT / ".env")
ZILLIZ_URI = os.getenv("ZILLIZ_CLOUD_URI")
ZILLIZ_KEY = os.getenv("ZILLIZ_CLOUD_API_KEY")

client = MilvusClient(
    uri=ZILLIZ_URI, 
    token=ZILLIZ_KEY
)

In [16]:
COLLECTION = "gri_units"
if COLLECTION in client.list_collections():
    client.drop_collection(COLLECTION)

schema = MilvusClient.create_schema(auto_id=False, enable_dynamic_field=True)
schema.add_field("unit_id", DataType.VARCHAR, max_length=64, is_primary=True)
schema.add_field("dense_embedding", DataType.FLOAT_VECTOR, dim=1024)
schema.add_field("sparse_embedding", DataType.SPARSE_FLOAT_VECTOR)

schema.add_field("knowledge_text", DataType.VARCHAR, max_length=4096)
schema.add_field("compliance_criteria", DataType.VARCHAR, max_length=2048)
schema.add_field("search_terms", DataType.VARCHAR, max_length=1024)
schema.add_field("banking_context", DataType.VARCHAR, max_length=2048)

index_params = client.prepare_index_params()
index_params.add_index(field_name="unit_id")
index_params.add_index(field_name="dense_embedding", index_type="AUTOINDEX", metric_type="COSINE")
index_params.add_index(field_name="sparse_embedding", index_type="SPARSE_INVERTED_INDEX", metric_type="IP")

client.create_collection(collection_name=COLLECTION, schema=schema, index_params=index_params)

In [ ]:
json_out = OUTPUT_DIR / "gri_units_with_knowledge.json"
with open(json_out, "r", encoding="utf-8") as f:
    units_data = json.load(f)

prepared = []
for unit in units_data:
    u = unit.copy()
    # Fix sparse embedding format
    sparse = u.get("sparse_embedding", {})
    if isinstance(sparse, dict):
        if "indices" in sparse and "values" in sparse:
            u["sparse_embedding"] = dict(zip(sparse["indices"], sparse["values"]))
            
    u["knowledge_text"] = str(u.get("knowledge_text", ""))[:4096]
    u["compliance_criteria"] = str(u.get("compliance_criteria", ""))[:2048]
    u["search_terms"] = json.dumps(u.get("search_terms", []))[:1024]
    u["banking_context"] = str(u.get("banking_context", ""))[:2048]
    u.setdefault("parent_requirement", "")
    prepared.append(u)

# Batch insert
batch_size = 100
for i in range(0, len(prepared), batch_size):
    batch = prepared[i : i + batch_size]
    client.insert(collection_name=COLLECTION, data=batch)